# 特定 10 组 Category-Location Pair 分析

分析 Base vs LoRA 模型在这10组pair上的表现

In [ ]:
import pandas as pd
df = pd.read_csv('uncommon_all_inference_results_base.csv')
print(f"Total time: {df['inference_time'].sum():.1f}s")
print(f"Avg per sample: {df['inference_time'].mean():.2f}s")

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.abspath(''), '..', '..'))
import config

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# 设置随机种子
np.random.seed(42)

In [ ]:
data_dir = config.FOCUS_DATA_DIR

# 定义要分析的10组pair
TARGET_PAIRS = [
    ('ship', 'grass'),
    ('car', 'water'),
    ('plane', 'water'),
    ('ship', 'indoors'),
    ('truck', 'water'),
    ('cat', 'water'),
    ('ship', 'forest'),
    ('bird', 'indoors'),
    ('ship', 'street'),
    ('plane', 'forest'),
]

TARGET_TOTAL = 300  # 目标总图片数

## 1. 加载数据并筛选

In [ ]:
# 加载详细结果
results_base = pd.read_csv(f'{data_dir}/uncommon_all_inference_results_base.csv')
results_lora = pd.read_csv(f'{data_dir}/uncommon_all_inference_results_lora.csv')

print(f"Base model total: {len(results_base)}")
print(f"LoRA model total: {len(results_lora)}")

In [ ]:
# 筛选目标pairs
def filter_target_pairs(df):
    mask = df.apply(lambda row: (row['category'], row['location']) in TARGET_PAIRS, axis=1)
    return df[mask].copy()

filtered_base = filter_target_pairs(results_base)
filtered_lora = filter_target_pairs(results_lora)

print(f"Filtered base: {len(filtered_base)}")
print(f"Filtered lora: {len(filtered_lora)}")

# 查看每个pair的数量
print("\nPer-pair counts (before sampling):")
pair_counts = filtered_base.groupby(['category', 'location']).size().reset_index(name='count')
for _, row in pair_counts.iterrows():
    print(f"  {row['category']:10s} - {row['location']:10s}: {row['count']}")
print(f"\nTotal: {pair_counts['count'].sum()}")

In [ ]:
# 计算需要从 plane-forest 中去除多少
current_total = len(filtered_base)
to_remove = current_total - TARGET_TOTAL

print(f"Current total: {current_total}")
print(f"Target total: {TARGET_TOTAL}")
print(f"Need to remove from plane-forest: {to_remove}")

# 从 plane-forest 随机采样保留的样本
plane_forest_base = filtered_base[(filtered_base['category'] == 'plane') & (filtered_base['location'] == 'forest')]
plane_forest_count = len(plane_forest_base)
keep_count = plane_forest_count - to_remove

print(f"\nplane-forest original: {plane_forest_count}")
print(f"plane-forest to keep: {keep_count}")

# 随机选择要保留的 plane-forest 样本的 filename
plane_forest_keep_filenames = plane_forest_base.sample(n=keep_count, random_state=42)['filename'].tolist()

# 过滤数据
def apply_sampling(df):
    # 保留非 plane-forest 的所有样本
    non_pf = df[~((df['category'] == 'plane') & (df['location'] == 'forest'))]
    # 保留选中的 plane-forest 样本
    pf_keep = df[(df['category'] == 'plane') & (df['location'] == 'forest') & 
                 (df['filename'].isin(plane_forest_keep_filenames))]
    return pd.concat([non_pf, pf_keep], ignore_index=True)

sampled_base = apply_sampling(filtered_base)
sampled_lora = apply_sampling(filtered_lora)

print(f"\nAfter sampling:")
print(f"  Base: {len(sampled_base)}")
print(f"  LoRA: {len(sampled_lora)}")

In [ ]:
# 验证每个pair的最终数量
print("Per-pair counts (after sampling):")
print("="*60)
final_counts = sampled_base.groupby(['category', 'location']).size().reset_index(name='count')
final_counts = final_counts.sort_values('count', ascending=False)
for _, row in final_counts.iterrows():
    print(f"  {row['category']:10s} - {row['location']:10s}: {row['count']}")
print("="*60)
print(f"Total: {final_counts['count'].sum()}")

## 2. 整体性能对比

In [ ]:
# 计算整体指标
def compute_metrics(df):
    valid = df[df['prediction'] != 'Unknown']
    total = len(df)
    valid_count = len(valid)
    unknown = total - valid_count
    
    correct = valid['correct'].sum()
    accuracy = correct / valid_count if valid_count > 0 else 0
    
    # 计算OOC预测率
    ooc_pred = (valid['prediction'] == 'Out-of-context').sum()
    ooc_rate = ooc_pred / valid_count if valid_count > 0 else 0
    
    return {
        'total': total,
        'valid': valid_count,
        'unknown': unknown,
        'correct': correct,
        'accuracy': accuracy,
        'ooc_pred': ooc_pred,
        'ooc_rate': ooc_rate
    }

metrics_base = compute_metrics(sampled_base)
metrics_lora = compute_metrics(sampled_lora)

print("="*70)
print("Overall Performance on 10 Target Pairs (300 samples)")
print("="*70)
print(f"{'Metric':<25} {'Base Model':>20} {'LoRA Model':>20}")
print("-"*70)
print(f"{'Total Samples':<25} {metrics_base['total']:>20} {metrics_lora['total']:>20}")
print(f"{'Valid Predictions':<25} {metrics_base['valid']:>20} {metrics_lora['valid']:>20}")
print(f"{'Unknown Predictions':<25} {metrics_base['unknown']:>20} {metrics_lora['unknown']:>20}")
print(f"{'Correct':<25} {metrics_base['correct']:>20} {metrics_lora['correct']:>20}")
print(f"{'Accuracy':<25} {metrics_base['accuracy']:>20.2%} {metrics_lora['accuracy']:>20.2%}")
print(f"{'OOC Predictions':<25} {metrics_base['ooc_pred']:>20} {metrics_lora['ooc_pred']:>20}")
print(f"{'OOC Rate':<25} {metrics_base['ooc_rate']:>20.2%} {metrics_lora['ooc_rate']:>20.2%}")
print("="*70)
print(f"\n📈 Accuracy Improvement: {metrics_lora['accuracy'] - metrics_base['accuracy']:+.2%}")

In [ ]:
# 可视化整体对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy对比
ax1 = axes[0]
models = ['Base Model', 'LoRA Model']
accuracies = [metrics_base['accuracy'], metrics_lora['accuracy']]
colors = ['steelblue', 'coral']
bars = ax1.bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy Comparison (300 samples)', fontsize=14)
ax1.set_ylim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax1.annotate(f'{acc:.2%}', xy=(bar.get_x() + bar.get_width()/2, acc),
                 xytext=(0, 5), textcoords='offset points', ha='center', fontsize=12, fontweight='bold')

# OOC Rate对比
ax2 = axes[1]
ooc_rates = [metrics_base['ooc_rate'], metrics_lora['ooc_rate']]
bars = ax2.bar(models, ooc_rates, color=colors, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Out-of-Context Prediction Rate')
ax2.set_title('OOC Prediction Rate Comparison', fontsize=14)
ax2.set_ylim(0, 1)
for bar, rate in zip(bars, ooc_rates):
    ax2.annotate(f'{rate:.2%}', xy=(bar.get_x() + bar.get_width()/2, rate),
                 xytext=(0, 5), textcoords='offset points', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Per-Pair 性能对比

In [ ]:
# 计算每个pair的统计
def compute_pair_stats(df):
    stats = df.groupby(['category', 'location']).agg(
        total=('prediction', 'count'),
        pred_ooc=('prediction', lambda x: (x == 'Out-of-context').sum()),
        pred_ic=('prediction', lambda x: (x == 'In-context').sum()),
        correct=('correct', 'sum')
    ).reset_index()
    stats['ooc_rate'] = stats['pred_ooc'] / stats['total']
    stats['accuracy'] = stats['correct'] / stats['total']
    stats['pair'] = stats['category'] + '-' + stats['location']
    return stats

stats_base = compute_pair_stats(sampled_base)
stats_lora = compute_pair_stats(sampled_lora)

# 合并对比
pair_comparison = stats_base[['pair', 'category', 'location', 'total', 'ooc_rate', 'accuracy']].merge(
    stats_lora[['pair', 'ooc_rate', 'accuracy']],
    on='pair',
    suffixes=('_base', '_lora')
)
pair_comparison['ooc_diff'] = pair_comparison['ooc_rate_lora'] - pair_comparison['ooc_rate_base']
pair_comparison['acc_diff'] = pair_comparison['accuracy_lora'] - pair_comparison['accuracy_base']

# 按accuracy差异排序
pair_comparison = pair_comparison.sort_values('acc_diff', ascending=False)

In [ ]:
# 显示详细对比表
print("="*100)
print("Per-Pair Performance Comparison (sorted by accuracy improvement)")
print("="*100)
print(f"{'Pair':<18} {'N':>6} {'OOC%(Base)':>12} {'OOC%(LoRA)':>12} {'OOC Diff':>10} {'Acc(Base)':>12} {'Acc(LoRA)':>12} {'Acc Diff':>10}")
print("-"*100)
for _, row in pair_comparison.iterrows():
    print(f"{row['pair']:<18} {row['total']:>6} {row['ooc_rate_base']:>12.2%} {row['ooc_rate_lora']:>12.2%} {row['ooc_diff']:>+10.2%} {row['accuracy_base']:>12.2%} {row['accuracy_lora']:>12.2%} {row['acc_diff']:>+10.2%}")
print("="*100)

In [ ]:
# Per-pair 准确率对比柱状图
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(pair_comparison))
width = 0.35

bars1 = ax.bar(x - width/2, pair_comparison['accuracy_base'], width, label='Base Model', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, pair_comparison['accuracy_lora'], width, label='LoRA Model', color='coral', edgecolor='black')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy by Category-Location Pair: Base vs LoRA', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(pair_comparison['pair'], rotation=45, ha='right', fontsize=10)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.15)

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.0%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.0%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

# 添加样本数标签
for i, (_, row) in enumerate(pair_comparison.iterrows()):
    ax.annotate(f'n={row["total"]}', xy=(i, 0.02), ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# OOC Rate 对比柱状图
fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - width/2, pair_comparison['ooc_rate_base'], width, label='Base Model', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, pair_comparison['ooc_rate_lora'], width, label='LoRA Model', color='coral', edgecolor='black')

ax.set_ylabel('Out-of-Context Prediction Rate', fontsize=12)
ax.set_title('OOC Prediction Rate by Pair: Base vs LoRA', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(pair_comparison['pair'], rotation=45, ha='right', fontsize=10)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.15)

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.0%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.0%}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 4. 预测一致性分析

In [ ]:
# 合并两个模型的预测
merged = sampled_base[['filename', 'category', 'location', 'ground_truth', 'prediction', 'correct']].merge(
    sampled_lora[['filename', 'prediction', 'correct']],
    on='filename',
    suffixes=('_base', '_lora')
)

merged['pair'] = merged['category'] + '-' + merged['location']
merged['both_correct'] = merged['correct_base'] & merged['correct_lora']
merged['both_wrong'] = ~merged['correct_base'] & ~merged['correct_lora']
merged['base_only_correct'] = merged['correct_base'] & ~merged['correct_lora']
merged['lora_only_correct'] = ~merged['correct_base'] & merged['correct_lora']

print("="*60)
print("Prediction Consistency Analysis (300 samples)")
print("="*60)
print(f"Both correct:       {merged['both_correct'].sum():>5} ({merged['both_correct'].mean():>6.2%})")
print(f"Both wrong:         {merged['both_wrong'].sum():>5} ({merged['both_wrong'].mean():>6.2%})")
print(f"Base only correct:  {merged['base_only_correct'].sum():>5} ({merged['base_only_correct'].mean():>6.2%})")
print(f"LoRA only correct:  {merged['lora_only_correct'].sum():>5} ({merged['lora_only_correct'].mean():>6.2%})")
print("="*60)
print(f"\n🎯 Net improvement (LoRA): {merged['lora_only_correct'].sum() - merged['base_only_correct'].sum():+d} samples")

In [ ]:
# 一致性饼图
fig, ax = plt.subplots(figsize=(8, 8))

sizes = [
    merged['both_correct'].sum(),
    merged['both_wrong'].sum(),
    merged['base_only_correct'].sum(),
    merged['lora_only_correct'].sum()
]
labels = [
    f'Both Correct\n({sizes[0]})',
    f'Both Wrong\n({sizes[1]})',
    f'Base Only\n({sizes[2]})',
    f'LoRA Only\n({sizes[3]})'
]
colors = ['#2ecc71', '#e74c3c', '#3498db', '#e67e22']
explode = (0.02, 0.02, 0.05, 0.05)

wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                   colors=colors, explode=explode, startangle=90)
ax.set_title('Prediction Consistency: Base vs LoRA (300 samples)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# 按pair统计一致性
consistency_by_pair = merged.groupby('pair').agg(
    total=('filename', 'count'),
    both_correct=('both_correct', 'sum'),
    both_wrong=('both_wrong', 'sum'),
    base_only=('base_only_correct', 'sum'),
    lora_only=('lora_only_correct', 'sum')
).reset_index()

consistency_by_pair['net_improvement'] = consistency_by_pair['lora_only'] - consistency_by_pair['base_only']
consistency_by_pair = consistency_by_pair.sort_values('net_improvement', ascending=False)

print("Consistency by Pair:")
print("="*90)
print(f"{'Pair':<18} {'N':>6} {'Both✓':>8} {'Both✗':>8} {'Base Only':>10} {'LoRA Only':>10} {'Net Δ':>8}")
print("-"*90)
for _, row in consistency_by_pair.iterrows():
    print(f"{row['pair']:<18} {row['total']:>6} {row['both_correct']:>8} {row['both_wrong']:>8} {row['base_only']:>10} {row['lora_only']:>10} {row['net_improvement']:>+8}")
print("="*90)

## 5. 每个 Pair 的图片可视化

In [ ]:
def visualize_pair_samples(pair_name, n_samples=6):
    """
    可视化某个pair的样本图片，显示两个模型的预测结果
    """
    pair_data = merged[merged['pair'] == pair_name]
    
    if len(pair_data) == 0:
        print(f"No data for pair: {pair_name}")
        return
    
    # 选择样本：优先选择有分歧的样本
    disagreements = pair_data[(pair_data['base_only_correct']) | (pair_data['lora_only_correct'])]
    agreements = pair_data[~((pair_data['base_only_correct']) | (pair_data['lora_only_correct']))]
    
    # 组合样本
    samples = pd.concat([
        disagreements.head(min(n_samples//2 + 1, len(disagreements))),
        agreements.head(n_samples - min(n_samples//2 + 1, len(disagreements)))
    ]).head(n_samples)
    
    if len(samples) == 0:
        samples = pair_data.head(n_samples)
    
    n_show = len(samples)
    cols = min(3, n_show)
    rows = (n_show + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes.reshape(1, -1)
    elif cols == 1:
        axes = axes.reshape(-1, 1)
    
    # 获取pair统计
    pair_stats = pair_comparison[pair_comparison['pair'] == pair_name].iloc[0]
    
    fig.suptitle(f"{pair_name} | N={pair_stats['total']} | Base Acc: {pair_stats['accuracy_base']:.0%} | LoRA Acc: {pair_stats['accuracy_lora']:.0%}", 
                 fontsize=14, fontweight='bold')
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        r, c = idx // cols, idx % cols
        ax = axes[r, c]
        
        # 加载图片
        img_path = os.path.join(data_dir, 'uncommon_images', row['filename'])
        try:
            img = Image.open(img_path)
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)
        
        # 设置标题
        base_pred = 'OOC' if row['prediction_base'] == 'Out-of-context' else 'IC'
        lora_pred = 'OOC' if row['prediction_lora'] == 'Out-of-context' else 'IC'
        base_correct = '✓' if row['correct_base'] else '✗'
        lora_correct = '✓' if row['correct_lora'] else '✗'
        
        # 颜色标记
        if row['both_correct']:
            title_color = 'green'
            status = 'Both ✓'
        elif row['both_wrong']:
            title_color = 'red'
            status = 'Both ✗'
        elif row['base_only_correct']:
            title_color = 'blue'
            status = 'Base Only ✓'
        else:
            title_color = 'orange'
            status = 'LoRA Only ✓'
        
        title = f"{row['filename'][:25]}...\nBase: {base_pred}{base_correct} | LoRA: {lora_pred}{lora_correct}\n[{status}]"
        ax.set_title(title, fontsize=9, color=title_color)
        ax.axis('off')
    
    # 隐藏空白子图
    for idx in range(n_show, rows * cols):
        r, c = idx // cols, idx % cols
        axes[r, c].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 可视化每个pair的样本
for pair in pair_comparison['pair'].tolist():
    print(f"\n{'='*80}")
    print(f"Visualizing: {pair}")
    print(f"{'='*80}")
    visualize_pair_samples(pair, n_samples=6)

## 6. 错误案例深入分析

In [ ]:
# LoRA改进的案例
lora_improved = merged[merged['lora_only_correct']]
print(f"\n🟢 LoRA Improved Cases: {len(lora_improved)}")
print("Distribution:")
print(lora_improved['pair'].value_counts())

In [ ]:
# LoRA退化的案例
lora_degraded = merged[merged['base_only_correct']]
print(f"\n🔴 LoRA Degraded Cases: {len(lora_degraded)}")
print("Distribution:")
print(lora_degraded['pair'].value_counts())

In [ ]:
# 两个模型都错的案例
both_wrong_cases = merged[merged['both_wrong']]
print(f"\n⚫ Both Models Wrong: {len(both_wrong_cases)}")
print("Distribution:")
print(both_wrong_cases['pair'].value_counts())

In [ ]:
# 可视化LoRA改进的案例
if len(lora_improved) > 0:
    print("\n" + "="*80)
    print("🟢 Sample images where LoRA improved (Base wrong, LoRA correct)")
    print("="*80)
    
    n_show = min(9, len(lora_improved))
    samples = lora_improved.head(n_show)
    
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4*rows))
    axes = axes.flatten() if n_show > 1 else [axes]
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        ax = axes[idx]
        img_path = os.path.join(data_dir, 'uncommon_images', row['filename'])
        try:
            img = Image.open(img_path)
            ax.imshow(img)
        except:
            pass
        ax.set_title(f"{row['pair']}\nBase: {row['prediction_base'][:3]}✗\nLoRA: {row['prediction_lora'][:3]}✓", 
                     fontsize=10, color='green')
        ax.axis('off')
    
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('LoRA Improved Cases', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# 可视化LoRA退化的案例
if len(lora_degraded) > 0:
    print("\n" + "="*80)
    print("🔴 Sample images where LoRA degraded (Base correct, LoRA wrong)")
    print("="*80)
    
    n_show = min(9, len(lora_degraded))
    samples = lora_degraded.head(n_show)
    
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4*rows))
    axes = axes.flatten() if n_show > 1 else [axes]
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        ax = axes[idx]
        img_path = os.path.join(data_dir, 'uncommon_images', row['filename'])
        try:
            img = Image.open(img_path)
            ax.imshow(img)
        except:
            pass
        ax.set_title(f"{row['pair']}\nBase: {row['prediction_base'][:3]}✓\nLoRA: {row['prediction_lora'][:3]}✗", 
                     fontsize=10, color='red')
        ax.axis('off')
    
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('LoRA Degraded Cases', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. 总结

In [ ]:
print("\n" + "="*80)
print("📊 SUMMARY: 10 Target Pairs Analysis (300 samples)")
print("="*80)

print(f"\n🎯 Overall Performance:")
print(f"   Base Model Accuracy: {metrics_base['accuracy']:.2%}")
print(f"   LoRA Model Accuracy: {metrics_lora['accuracy']:.2%}")
print(f"   Improvement: {metrics_lora['accuracy'] - metrics_base['accuracy']:+.2%}")

print(f"\n📈 OOC Prediction Rate:")
print(f"   Base Model: {metrics_base['ooc_rate']:.2%}")
print(f"   LoRA Model: {metrics_lora['ooc_rate']:.2%}")
print(f"   Change: {metrics_lora['ooc_rate'] - metrics_base['ooc_rate']:+.2%}")

print(f"\n🔄 Prediction Consistency:")
print(f"   Both Correct: {merged['both_correct'].sum()} ({merged['both_correct'].mean():.1%})")
print(f"   Both Wrong: {merged['both_wrong'].sum()} ({merged['both_wrong'].mean():.1%})")
print(f"   LoRA Improved: {merged['lora_only_correct'].sum()}")
print(f"   LoRA Degraded: {merged['base_only_correct'].sum()}")
print(f"   Net Improvement: {merged['lora_only_correct'].sum() - merged['base_only_correct'].sum():+d}")

print(f"\n🏆 Best Performing Pairs (LoRA):")
top_pairs = pair_comparison.nlargest(3, 'accuracy_lora')
for _, row in top_pairs.iterrows():
    print(f"   {row['pair']}: {row['accuracy_lora']:.0%}")

print(f"\n⚠️ Most Challenging Pairs (LoRA):")
bottom_pairs = pair_comparison.nsmallest(3, 'accuracy_lora')
for _, row in bottom_pairs.iterrows():
    print(f"   {row['pair']}: {row['accuracy_lora']:.0%}")

print(f"\n📊 Pairs with Biggest LoRA Improvement:")
most_improved = pair_comparison.nlargest(3, 'acc_diff')
for _, row in most_improved.iterrows():
    print(f"   {row['pair']}: {row['accuracy_base']:.0%} → {row['accuracy_lora']:.0%} ({row['acc_diff']:+.0%})")

print("\n" + "="*80)

In [ ]:
# 所有300个文件名
all_filenames = sampled_base['filename'].tolist()
print(f"Total: {len(all_filenames)}")
print(all_filenames)

# LoRA improved 的文件名
lora_improved_filenames = merged[merged['lora_only_correct']]['filename'].tolist()
print(f"\nLoRA Improved: {len(lora_improved_filenames)}")
print(lora_improved_filenames)

# LoRA degraded 的文件名
lora_degraded_filenames = merged[merged['base_only_correct']]['filename'].tolist()
print(f"\nLoRA Degraded: {len(lora_degraded_filenames)}")
print(lora_degraded_filenames)

# Both wrong 的文件名
both_wrong_filenames = merged[merged['both_wrong']]['filename'].tolist()
print(f"\nBoth Wrong: {len(both_wrong_filenames)}")
print(both_wrong_filenames)